# Laya — automatic labelling (Sonnet)

New lectures → speech-to-text → engine v4 → **Sonnet labels every finding** (same guide and cards as the blind test, where it matched you on 58 of 60 cards) → you spot-check a sample.

| Lectures | Why |
|---|---|
| **YouTube**, 10 sheikhs, 2 short lectures each | the real use of the app: no written text, different voices and styles |
| **Bin Baz**, 5 new short lessons | written text = ground truth: findings on a marked quote are skipped (their automatic label is right 99 % of the time), so Sonnet's work goes where the negatives are |

Lectures, transcripts and results are kept in `data/laya_datagen/`: `lectures/<id>/` (meta, transcript, words with timestamps), `labels/sonnet.json` (every answer with Sonnet's reason), `labels/sonnet_runs.jsonl` (time and usage of every call).

## Setup

In [7]:
import sys, os, importlib
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "backend").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from IPython.display import display
from backend.laya_datagen import config, asr as asr_mod, pipeline, cards, lectures, autolabel, agent_eval
from backend.laya_datagen.engine_bridge import EngineBridge
from backend.laya_datagen.sources import binbaz, youtube
for _m in (cards, agent_eval, autolabel, lectures): importlib.reload(_m)   # picks up code edits without restarting the kernel

ENGINE = "v4"
HADITH_CORPUS = str(ROOT / "data/corpora/hadith_open_hadith_data.json")
RUN_MODELS = True       # True: Sonnet labels new cards (uses your Claude allowance) · False: show saved results only (safe for Run all)

## Lectures
The list is in `backend/laya_datagen/lectures.py` (add video ids or Bin Baz lessons there). Downloads and transcription (Groq) happen once; everything is cached.

In [8]:
bridge = EngineBridge(hadith_path=HADITH_CORPUS); bridge.start()
bundles, rows = {}, []

def transcribe(audio):
    return asr_mod.transcribe(audio, backend="groq", model="whisper-large-v3-turbo")

for sheikh, vids in lectures.YOUTUBE.items():
    for vid in vids:
        try:
            v = youtube.download(vid, sheikh)
            a = transcribe(youtube.AUDIO_DIR / f"{v.id}.mp3")
            youtube.save_transcript(v.id, a)
        except Exception as e:                       # e.g. Groq rate limit: re-run later, the rest is cached
            print(f"✘ {sheikh} {vid}: {type(e).__name__}: {str(e)[:120]}"); continue
        b = pipeline.process_no_gt(v.id, v.title, v.url, a, bridge, [ENGINE])
        bundles[v.id] = b
        rows.append({"source": "YouTube", "sheikh": sheikh, "lecture": v.title[:50], "minutes": round(v.duration / 60),
                     "findings": len(b["records"]), "on a GT quote (skipped)": "—"})
for url in lectures.BINBAZ:
    les = binbaz.get_lesson(url)
    try:
        a = transcribe(binbaz.download_mp3(les))
        youtube.save_transcript(les.id, a, {"id": les.id, "url": les.url, "title": les.title, "sheikh": "bin_baz"})
    except Exception as e:
        print(f"✘ {les.id}: {type(e).__name__}: {str(e)[:120]}"); continue
    b = pipeline.process_cached(les, a, bridge, [ENGINE])
    if not b["report"]["passed_gate"]:
        print(f"✘ {les.id}: written text and speech don't match well enough"); continue
    bundles[les.id] = b
    rows.append({"source": "Bin Baz", "sheikh": "bin_baz", "lecture": les.title[:50], "minutes": round(a["words"][-1]["end"] / 60),
                 "findings": len(b["records"]), "on a GT quote (skipped)": sum(autolabel.is_gt_finding(r) for r in b["records"])})
df = pd.DataFrame(rows)
display(df)
print(f"{len(bundles)} lectures · {df.minutes.sum() / 60:.1f} h of audio · {df.findings.sum()} findings")

,source,sheikh,lecture,minutes,findings,on a GT quote (skipped)
0,YouTube,ibn_uthaymeen,لقاء[67 من 219] شرح ومناقشة آية الكرسي 1 - الش...,21,20,—
1,YouTube,ibn_uthaymeen,471- شرح رياض الصالحين / باب الاستغفار / الاسـ...,25,63,—
2,YouTube,al_fawzan,شرح كتاب القواعد الأربعة كاملاً | الشيخ صالح ا...,34,52,—
3,YouTube,al_fawzan,حذرنا النبيﷺ من فتنة لا تسمع ولا تنطق ولا ترى!...,23,27,—
4,YouTube,aziz_farhan_al_anzi,محاضرة بعنوان: وأفوض أمري إلى الله | فضيلة الش...,33,30,—
5,YouTube,aziz_farhan_al_anzi,خطوات علاج القلق و الاكتئاب لفضيلة الشيخ الدكت...,34,45,—
6,YouTube,al_albani,معنى كلمة التوحيد لا إله إلا الله وبيان بعض نو...,30,19,—
7,YouTube,al_albani,موعظة بليغة في تقوى الله عز وجل - الشيخ الالبا...,30,23,—
8,YouTube,saleh_al_sheikh,موضوع سورة العنكبوت ومقصدها | الشيخ صالح آل الشيخ,22,44,—
9,YouTube,saleh_al_sheikh,60 شرح كتاب التوحيد ( باب ماجاء في المصورين )...,15,16,—


25 lectures · 9.2 h of audio · 764 findings


## The pool
Every finding except the ones on a quote marked in Bin Baz's written text, in a fixed shuffled order (lectures and sheikhs mixed in every batch).

In [9]:
pool = autolabel.pool(bundles, ENGINE)
src = pd.Series(["YouTube" if r["doc_id"].startswith("yt_") else "Bin Baz" for r in pool]).value_counts()
print(len(pool), "findings to label:", dict(src))

717 findings to label: {'YouTube': np.int64(665), 'Bin Baz': np.int64(52)}


## Run Sonnet
Sonnet (medium effort) through `claude -p`, 10 cards per call, from `~/laya_labeller` with every tool off. Re-running continues with the next unlabelled cards; if your usage limit is reached it stops cleanly and you re-run later.

In [10]:
N_CARDS = 320                # how many NEW cards this run labels
autolabel.run(pool, bundles, N_CARDS, RUN_MODELS)

batch 1/32 · 10/10 answered · 16 s · 2,061 tokens out · ≈$0.051
batch 2/32 · 10/10 answered · 9 s · 1,057 tokens out · ≈$0.044
batch 3/32 · 10/10 answered · 9 s · 924 tokens out · ≈$0.046
batch 4/32 · 10/10 answered · 16 s · 1,828 tokens out · ≈$0.051
batch 5/32 · 10/10 answered · 9 s · 907 tokens out · ≈$0.043
batch 6/32 · 10/10 answered · 17 s · 2,332 tokens out · ≈$0.061
batch 7/32 · 10/10 answered · 16 s · 2,250 tokens out · ≈$0.060
batch 8/32 · 10/10 answered · 18 s · 2,151 tokens out · ≈$0.055
batch 9/32 · 10/10 answered · 20 s · 2,424 tokens out · ≈$0.058
batch 10/32 · 10/10 answered · 19 s · 2,590 tokens out · ≈$0.065
batch 11/32 · 10/10 answered · 13 s · 1,742 tokens out · ≈$0.052
batch 12/32 · 10/10 answered · 10 s · 1,044 tokens out · ≈$0.050
batch 13/32 · 10/10 answered · 20 s · 2,824 tokens out · ≈$0.065
batch 14/32 · 10/10 answered · 15 s · 1,860 tokens out · ≈$0.050
batch 15/32 · 10/10 answered · 9 s · 1,087 tokens out · ≈$0.052
batch 16/32 · 10/10 answered · 11 s · 1,29

## Results

In [11]:
s = autolabel.summary(pool, bundles)
print(f"{s['labelled']} of {s['pool']} findings labelled")
display(pd.DataFrame(s["by source"]).fillna(0).astype(int))
pd.Series(s["usage"])

717 of 717 findings labelled


,YouTube,Bin Baz
✔ Real quote · Complete,503,23
✘ Not a quote,63,9
✔ Real quote · Missing start,40,5
≈ Quoted by meaning,20,2
✔ Real quote · Missing end,32,11
📖 Ayah inside a hadith · Complete,3,0
✔ Real quote · Missing both,3,2
📖 Ayah inside a hadith · Missing start,1,0


calls                                                     72
cards answered                                           717
time                               16.9 min (1.4 s per card)
tokens out                                           126,809
tokens in (cache write / read)             635,711 / 251,098
API-equivalent cost               $3.91 ($5 per 1,000 cards)
dtype: object

## Spot check
Every 7th labelled card with Sonnet's answer and reason. One click: **Sonnet is right / wrong / not sure**, then the next card. Saved to `labels/sonnet_check.json`.

In [12]:
autolabel.spot_check(pool, bundles)

---
*Next (after you check the first batch): more cards / lectures, then export to Laya training rows.*